**Imports & Setup**

This cell imports all the necessary Python libraries required for the project, including Pandas for data manipulation, Psycopg2 for PostgreSQL database connectivity, Plotly for interactive data visualizations, and Yagmail for sending automated emails.

In [8]:
#Importing libraries
import pandas as pd
import psycopg2
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
import numpy
from datetime import date
import yagmail
from sqlalchemy import create_engine, URL

pio.renderers.default = "iframe"

**Database Connection**

This cell establishes a connection to the PostgreSQL database (`Policy-Management-System`) using SQLAlchemy. It creates an engine with the necessary credentials and assigns it to `myconn`, which is used throughout the notebook for executing SQL queries and fetching data into Pandas DataFrames.

In [ ]:
# Create the URL object which safely encodes passwords with special characters
url_object = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password="your-server-password",  # Your plain password here
    host="localhost",
    port=5432,
    database="Policy-Management-System",
)

# Create the engine using the safe URL
engine = create_engine(
    url_object,
    connect_args={'options': '-csearch_path=insurance'}
)

# Assign it to myconn so your existing pd.read_sql calls work
myconn = engine


**Database Initialization**

This cell defines a function that automatically executes the DDL and DML SQL scripts to create the necessary tables and insert initial data if they are missing. This ensures the database is set up and ready for queries before proceeding with any analysis.

In [ ]:
def initialize_database(engine):
    """
    Checks if the database tables exist and are populated.
    If not, executes the DDL and DML scripts to set them up.
    """
    raw_conn = engine.raw_connection()
    try:
        with raw_conn.cursor() as cursor:
            # Check if a core table ('company') exists in the 'insurance' schema
            cursor.execute("SELECT EXISTS (SELECT FROM pg_tables WHERE schemaname = 'insurance' AND tablename = 'company');")
            exists = cursor.fetchone()[0]
            
            if not exists:
                print("Tables not found. Initializing database with DDL and DML...")
                
                # Execute DDL
                with open(r"f:\Projects\DBMS\Phase 1-Legacy_Simple_DBMS\Insurance database DDL.sql", "r", encoding="utf-8") as ddl_file:
                    cursor.execute(ddl_file.read())
                    
                # Execute DML
                with open(r"f:\Projects\DBMS\Phase 1-Legacy_Simple_DBMS\Insurance database DML.sql", "r", encoding="utf-8") as dml_file:
                    cursor.execute(dml_file.read())
                    
                raw_conn.commit()
                print("Database initialized successfully.")
            else:
                # Tables exist, check if data exists
                cursor.execute("SELECT COUNT(*) FROM insurance.company;")
                count = cursor.fetchone()[0]
                if count == 0:
                    print("Tables exist but no data found. Inserting data with DML...")
                    with open(r"f:\Projects\DBMS\Phase 1-Legacy_Simple_DBMS\Insurance database DML.sql", "r", encoding="utf-8") as dml_file:
                        cursor.execute(dml_file.read())
                    raw_conn.commit()
                    print("Data inserted successfully.")
                else:
                    print("Database already initialized and contains data.")
    except Exception as e:
        raw_conn.rollback()
        print(f"Error during initialization: {e}")
    finally:
        raw_conn.close()

# Automatically initialize database when running this notebook
initialize_database(engine)

## Visualization

**Customer and Agent Distribution**

This cell executes a SQL query to count the number of distinct customers and agents in each city. It then fetches the result into a Pandas DataFrame and creates an interactive grouped bar chart using Plotly to visualize the distribution of customers and employees across different cities.

In [10]:
#count of customers and agents
query="""
select c.city, count(distinct c.Customer_id) as num_of_customers, count(distinct a.Agent_id) as num_of_agents 
from customer c 
left join agent a on c.city = a.city
group by c.city
order by num_of_customers desc, num_of_agents desc
"""

df = pd.read_sql(query, myconn)

plot = go.Figure(data=[go.Bar(
    name = 'No. of Customers',
    x = df['city'],
    y = df['num_of_customers']),
    go.Bar(
    name = 'No. of Agents',
    x = df['city'],
    y = df['num_of_agents']
   )
])

plot.update_layout(title="Number of Customers and Number of Employees in each City", xaxis_title="City", yaxis_title="Count")
display(HTML(plot.to_html(include_plotlyjs='cdn', full_html=False)))


**Year-wise Policy and Revenue Trend**

This cell performs a year-wise analysis by querying the database for the total number of policies issued and the total revenue generated per year. It extracts the year from the policy start dates and uses Plotly to plot a dual-line graph, showing how policy counts and revenue have changed over the years.

In [11]:
#year-wise analysis
query="""
select extract(year from p.start_date)::int as policy_year, count(p.Policy_no) as Num_of_policy, sum(pa.amount) as revenue
from policy p, customer_policy cp, payment pa
where p.Policy_no = cp.Policy_no and cp.Customer_id = pa.Customer_id
group by policy_year
"""

df_temp = pd.read_sql(query, myconn)
df_temp.columns = [column.lower() for column in df_temp.columns]
df_temp['policy_year'] = pd.to_datetime(df_temp['policy_year'], format='%Y')
df_temp['revenue_in_thousands'] = df_temp['revenue'] / 1000

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_temp['policy_year'], y=df_temp['num_of_policy'], mode='lines+markers', name='Num of policy'))
fig.add_trace(go.Scatter(x=df_temp['policy_year'], y=df_temp['revenue_in_thousands'], mode='lines+markers', name='Revenue (in thousands)'))
fig.update_layout(title='Change in Number of Policies and Revenue(in thousands) over the years', xaxis_title='Policy Year', yaxis_title='Value')

display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))


**Age Group Demographics**

This cell categorizes customers into different age groups based on their date of birth (DOB) and counts the total number of policies for each group. The results are then visualized using a Plotly pie chart to show the proportion of policies held by respective age groups.

In [12]:
#No.of Policies Taken by Respective Age Group
query="""
select age_group, count(Policy_no) as num_of_policy
from 
( select 
case when extract(year from age(current_date, c.DoB)) >= 18 and extract(year from age(current_date, c.DoB)) < 31 then '18-30'
when extract(year from age(current_date, c.DoB)) >= 31 and extract(year from age(current_date, c.DoB)) < 41 then '31-40'
when extract(year from age(current_date, c.DoB)) >= 41 and extract(year from age(current_date, c.DoB)) < 51 then '41-50'
when extract(year from age(current_date, c.DoB)) >= 51 and extract(year from age(current_date, c.DoB)) < 61 then '51-60'
when extract(year from age(current_date, c.DoB)) >= 61 and extract(year from age(current_date, c.DoB)) < 71 then '61-70'
when extract(year from age(current_date, c.DoB)) >= 71 and extract(year from age(current_date, c.DoB)) < 81 then '71-80'
when extract(year from age(current_date, c.DoB)) >= 81 and extract(year from age(current_date, c.DoB)) < 91 then '81-90'
end as age_group, p.policy_no, c.DoB
from policy p, customer_policy cp, customer c
where p.Policy_no = cp.Policy_no and cp.Customer_id = c.Customer_id)d
group by age_group
"""

df2 = pd.read_sql(query, myconn)

fig = px.pie(df2, values='num_of_policy', names='age_group', title='No.of Policies Taken by Each Age Group')
display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))


**Insurance Plan Categorization**

This cell analyzes insurance plans by categorizing health, car, and home policies into `basic`, `silver`, `gold`, and `platinum` plans based on their premium amounts. It calculates the total number of policies and premiums for each category and plan, merging them into a comprehensive pivot table summary.

In [13]:
query="""
select health_plan,count(Policy_no) as num_of_policy, sum(premium) as premium
from 
( select 
case when premium>= 1000 and premium<2000 then 'basic plan'
when premium>=2000 and premium<3500 then 'silver plan'
when premium>=3500 and premium<4500 then 'gold plan'
when premium>=4500 then 'platinum plan'
end as health_plan,p.policy_no,p.premium 
from policy p, health h 
where p.Policy_no=h.Policy_no)d
group by health_plan
"""

df_h = pd.read_sql(query, myconn)

query_c="""
select car_plan,count(Policy_no) as num_of_policy, sum(premium) as premium
from 
( select 
case when premium>= 1000 and premium<1500 then 'basic plan'
when premium>1500 and premium<=2000 then 'silver plan'
when premium>2000 and premium<=3000 then 'gold plan'
when premium>3000 then 'platinum plan'
end as car_plan,p.policy_no,p.premium 
from policy p, car c 
where p.Policy_no=c.Policy_no)d
group by car_plan
"""

df_c = pd.read_sql(query_c, myconn)

query_ho="""
select home_plan,count(Policy_no) as num_of_policy, sum(premium) as premium
from 
( select 
case when premium>= 4000 and premium<5000 then 'basic plan'
when premium>=5000 and premium<9000 then 'silver plan'
when premium>=9000 and premium<12000 then 'gold plan'
when premium>=12000 then 'platinum plan'
end as home_plan,p.policy_no,p.premium 
from policy p, home ho 
where p.Policy_no=ho.Policy_no)d
group by home_plan
"""

df_ho = pd.read_sql(query_ho, myconn)

df_h.rename(columns = {'health_plan':'plan'}, inplace = True)
df_c.rename(columns = {'car_plan':'plan'}, inplace = True)
df_ho.rename(columns = {'home_plan':'plan'}, inplace = True)

df_h['Category'] = "health"
df_c['Category'] = "car"
df_ho['Category'] = "home"

df_ho_g=df_ho.groupby(['Category','plan']).sum()
df_h_g=df_h.groupby(['Category','plan']).sum()
df_c_g=df_c.groupby(['Category','plan']).sum()

df_merge = pd.concat([df_h_g, df_c_g, df_ho_g])

output = pd.pivot_table(data=df_merge, 
                        index=['plan'], 
                        columns=['Category'], 
                        values='num_of_policy',
                        aggfunc='sum')
output

Category,car,health,home
plan,,,
basic plan,31,55,18
gold plan,28,41,20
platinum plan,17,55,45
silver plan,27,53,19


**Dynamic Visualization for Insurance Plans**

This cell defines a custom function `multi_plot` that takes the previously generated pivot table data and creates an interactive, grouped bar chart with dropdown menus. This allows users to dynamically filter and view the number of policies for each plan across different insurance types (car, health, home).

In [14]:
#Number of policies for each plan per insurance type

def multi_plot(df, title, addAll = True):
    fig = go.Figure()

    for column in df.columns.to_list():
        fig.add_trace(
            go.Bar(
                x = df.index,
                y = df[column],
                name = column
            )
        )

    button_all = dict(label = 'All',
                      method = 'update',
                      args = [{'visible': df.columns.isin(df.columns),
                               'title': 'All',
                               'showlegend':True}])

    def create_layout_button(column):
        return dict(label = column,
                    method = 'update',
                    args = [{'visible': df.columns.isin([column]),
                             'title': column,
                             'showlegend': True}])

    fig.update_layout(
        updatemenus=[go.layout.Updatemenu(
            active = 0,
            buttons = ([button_all] * addAll) + list(df.columns.map(lambda column: create_layout_button(column)))
            )
        ],
         yaxis_type="log"       
    )
    # Update remaining layout properties
    fig.update_layout(
        title_text=title,
        height=600,
        xaxis_title="Plan", 
        yaxis_title="Count"
        
    )
   
    display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))
    
multi_plot(output, title="Number of policies for each plan per insurance type") 


## Features

**Automated Birthday Emails**

This cell checks if today is any customer's birthday by querying the database for customer dates of birth and emails. If a match is found, it automatically sends a customized `Happy Birthday` greeting email to the respective customer using the `yagmail` library.

In [ ]:

# today = date.today()
# d1 = today.month
# d2 = today.day

# Message="""Wish you happy birthday from Insurance_Northeastern!!

# Regards
# Team Insurance_Northeastern """

#password = input("Type your password and press enter: ")

# yag = yagmail.SMTP('karakata.s@northeastern.edu', 'Northeastern@2021', host='smtp.office365.com', port=587, smtp_starttls=True, smtp_ssl=False)
# query="Select dob,email from insurance.Customer;"

# df_f = pd.read_sql(query, myconn)

# for i in range(len(df_f)):
#     m = df_f.loc[i, "dob"].month
#     d = df_f.loc[i, "dob"].day
#     e = df_f.loc[i, "email"]
#     if m == d1 and d == d2:
#         yag.send(
#             to=e,
#             subject="Happy Birthday!!",
#             contents=Message, 
#         )

**Interactive Database Insertion**

This cell provides an interactive command-line interface to insert new records into the database. It prompts the user to select a table (Payment, Customer, or Policy) and takes the required data inputs to execute the corresponding SQL `INSERT` statement using the `psycopg2` library.

In [ ]:
#Creating a cursor object using the cursor() method
conn = psycopg2.connect(host='localhost', user='postgres', password='your-server-password', dbname='Policy-Management-System', port='5432')
cursor = conn.cursor()

with conn.cursor() as schema_cursor:
    schema_cursor.execute("set search_path to insurance;")

table = input("""Please select the table number on which you want to insert the data

1.Payment
2.Customer
3.Policy
: """).strip()

if table not in {'1', '2', '3'}:
    print("No table selected. Insert skipped.")
else:
    # Preparing SQL query to INSERT a record into the database.
    if table == '1':
        insert_stmt = (
            """insert into payment(billing_id, bank_account_no, payment_Date, amount, customer_id)
            VALUES (%s, %s, %s, %s, %s)"""
        )

        billing_id = input("Type your billing_id and press enter: ")
        bank_account_no = input("Type your bank_account_no and press enter: ")
        payment_Date = input("Type your payment_Date and press enter: ")
        amount = input("Type your amount and press enter: ")
        customer_id = input("Type your customer_id and press enter: ")
        data = (billing_id, bank_account_no, payment_Date, amount, customer_id)

    elif table == '2':
        insert_stmt = (
            """insert into customer(Customer_id, DOB, First_name, Last_name, Phone_num, email, City, Zipcode, Agent_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)"""
        )
        Customer_id = input("Type your Customer_id and press enter: ")
        DOB = input("Type your DOB and press enter: ")
        First_name = input("Type your First_name and press enter: ")
        Last_name = input("Type your Last_name and press enter: ")
        Phone_num = input("Type your Phone_num and press enter: ")
        email = input("Type your email and press enter: ")
        City = input("Type your City and press enter: ")
        Zipcode = input("Type your Zipcode and press enter: ")
        Agent_id = input("Type your Agent_id and press enter: ")
        data = (Customer_id, DOB, First_name, Last_name, Phone_num, email, City, Zipcode, Agent_id)

    else:
        insert_stmt = (
            """insert into policy(Policy_no, Start_date, End_date, Premium, Coverage, Branch_id, Agent_id)
            VALUES (%s, %s, %s, %s, %s, %s, %s)"""
        )
        Policy_no = input("Type your Policy_no and press enter: ")
        Start_date = input("Type your Start_date and press enter: ")
        End_date = input("Type your End_date and press enter: ")
        Premium = input("Type your Premium and press enter: ")
        Coverage = input("Type your Coverage and press enter: ")
        Branch_id = input("Type your Branch_id and press enter: ")
        Agent_id = input("Type your Agent_id and press enter: ")
        data = (Policy_no, Start_date, End_date, Premium, Coverage, Branch_id, Agent_id)

    if any(value == "" for value in data):
        print("No values entered. Insert skipped.")
    else:
        try:
            cursor.execute(insert_stmt, data)
            conn.commit()
            print('Thank you, Data is now entered')
        except psycopg2.Error as e:
            conn.rollback()
            print("Database Error: Failed to insert data. This is likely because a referenced ID (like Agent_id or Branch_id) you entered doesn't exist.")
            print(f"Details: {e}")

Database Error: Failed to insert data. This is likely because a referenced ID (like Agent_id or Branch_id) you entered doesn't exist.
Details: insert or update on table "customer" violates foreign key constraint "customer_agent_id_fkey"
DETAIL:  Key (agent_id)=(1212) is not present in table "agent".



**DDL and DML File Links**

This cell uses IPython display utilities and the `pathlib` library to reference the DDL (Data Definition Language) and DML (Data Manipulation Language) SQL script files associated with the database project, defining their paths on the local system.

In [ ]:
from pathlib import Path
from IPython.display import display

base_dir = Path(r"f:\Projects\DBMS")
ddl_path = base_dir / "Insurance database DDL.sql"
dml_path = base_dir / "Insurance database DML.sql"
dql_path = base_dir / "Insurance database DQL.sql"

runner_conn = psycopg2.connect(
    host="localhost",
    user="postgres",
    password="your-server-password",
    dbname="Policy-Management-System",
    port="5432",
)
runner_conn.autocommit = True
with runner_conn.cursor() as schema_cursor:
    schema_cursor.execute("set search_path to insurance;")


def strip_sql_comments(sql_text):
    cleaned_lines = []
    for line in sql_text.splitlines():
        stripped_line = line.lstrip()
        if stripped_line.startswith("--"):
            continue
        if "--" in line:
            line = line.split("--", 1)[0]
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines)


def split_sql_statements(sql_text):
    statements = []
    buffer = []
    in_single_quote = False
    in_dollar_quote = False
    index = 0

    while index < len(sql_text):
        character = sql_text[index]
        next_two = sql_text[index:index + 2]

        if not in_single_quote and next_two == "$$":
            in_dollar_quote = not in_dollar_quote
            buffer.append("$$")
            index += 2
            continue

        if not in_dollar_quote and character == "'":
            buffer.append(character)
            if in_single_quote and index + 1 < len(sql_text) and sql_text[index + 1] == "'":
                buffer.append("'")
                index += 2
                continue
            in_single_quote = not in_single_quote
            index += 1
            continue

        if not in_single_quote and not in_dollar_quote and character == ";":
            statement = "".join(buffer).strip()
            if statement:
                statements.append(statement)
            buffer = []
            index += 1
            continue

        buffer.append(character)
        index += 1

    final_statement = "".join(buffer).strip()
    if final_statement:
        statements.append(final_statement)

    return statements


def run_sql_file(sql_path, connection, show_query_results=False):
    sql_text = strip_sql_comments(sql_path.read_text(encoding="utf-8")).lstrip("\ufeff")
    statements = split_sql_statements(sql_text)
    cursor = connection.cursor()

    print(f"Running {sql_path.name} ...")
    for statement_index, statement in enumerate(statements, start=1):
        cleaned_statement = statement.strip().lstrip("\ufeff")
        if not cleaned_statement:
            continue

        statement_head = cleaned_statement.lstrip().lower()
        if show_query_results and statement_head.startswith(("select", "with")):
            print(f"\n{sql_path.stem} - query {statement_index}")
            result_frame = pd.read_sql_query(cleaned_statement, connection)
            display(result_frame)
        else:
            cursor.execute(cleaned_statement)

    print(f"Finished {sql_path.name}")


runner_cursor = runner_conn.cursor()
runner_cursor.execute("drop schema if exists insurance cascade;")
runner_conn.commit()

run_sql_file(ddl_path, runner_conn)
runner_cursor.execute("drop trigger if exists payment_check on payment;")
runner_conn.commit()

run_sql_file(dml_path, runner_conn)
runner_cursor.execute("create trigger payment_check before insert on payment for each row execute function payment_check_fn();")
runner_conn.commit()

run_sql_file(dql_path, runner_conn, show_query_results=True)
